# Day 208 — Week 3 Day 1: LoRA / QLoRA Fundamentals & PEFT Setup
### Parameter-Efficient Fine-Tuning for TeleServe India Ticket Classification

---

**Curriculum position:** Month 13, Day 208 (Week 3 of 4: LoRA/QLoRA, Days 208–211)
**Builds on:** Month 9 DistilBERT fine-tuning (Days 164–167, ReviewPulse India, `AutoModelForSequenceClassification`)
**New scenario:** TeleServe India support ticket **priority classification** (P1/P2/P3) — same 3-class shape as Month 9, new domain
**Environment:** Google Colab, **T4 GPU** (Runtime → Change runtime type → T4 GPU) for Days 209–211; today's tasks run fine on CPU too
**Total points:** 100 + 10★ bonus

---

### Why this day exists

Month 9 taught you to fine-tune DistilBERT the standard way: **every one of its ~67 million parameters gets updated**. That works, but it doesn't scale — every new task means a new 260 MB checkpoint, and on your 8GB RAM / MX130 laptop (or even free-tier Colab), fine-tuning a 7B+ LLM this way is simply not possible.

**LoRA (Low-Rank Adaptation)** freezes the entire pretrained model and injects small trainable "adapter" matrices into a handful of layers. **QLoRA** goes further — it loads the frozen base model in 4-bit precision, so you can fine-tune models far larger than your GPU's VRAM would otherwise allow.

Today is **fundamentals + setup only** — no training yet. You will verify every claim about LoRA (parameter counts, what freezes, what trains, why it starts as a no-op) by running real code against a real DistilBERT model, the same way you verified `InvalidUpdateError` and `RetryPolicy` behavior in Days 200–202. Day 209 uses what you build today to actually fine-tune.


---
## Section 1: Raw Data (locked — do not modify)

This section fixes the facts you'll build against: the frozen base model's architecture, and a small locked sample of TeleServe India ticket text (seed=208) used only to test that forward/backward passes work — not for training.


In [1]:
# ============================================================
# RAW DATA — LOCKED. Do not edit this cell's outputs/values.
# Model architecture facts (verified by running this exact code)
# and a fixed sample of ticket text, seed=208.
# ============================================================
import random
SEED = 208
random.seed(SEED)

MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 3                      # P1 (urgent), P2 (medium), P3 (low) — same 3-class shape as Month 9's 3-class sentiment
LABEL_NAMES = {0: "P1_urgent", 1: "P2_medium", 2: "P3_low"}

# Locked sample tickets — TeleServe India, seed=208 (for forward/backward pass testing only)
SAMPLE_TICKETS = [
    ("My internet has been down for 6 hours, this is urgent!", 0),   # P1
    ("Router blinking red, no connectivity since morning, please fix ASAP", 0),  # P1
    ("Billing amount seems slightly higher this month, can you check?", 1),      # P2
    ("Would like to upgrade my plan sometime next week", 1),                    # P2
    ("Just checking my current data balance please.", 2),                       # P3
    ("Where can I download the invoice PDF from the app?", 2),                  # P3
]

print(f"Model: {MODEL_NAME}")
print(f"Task: {NUM_LABELS}-class sequence classification -> {LABEL_NAMES}")
print(f"Sample tickets (seed={SEED}): {len(SAMPLE_TICKETS)} rows")
for text, label in SAMPLE_TICKETS:
    print(f"  [{LABEL_NAMES[label]}] {text}")


Model: distilbert-base-uncased
Task: 3-class sequence classification -> {0: 'P1_urgent', 1: 'P2_medium', 2: 'P3_low'}
Sample tickets (seed=208): 6 rows
  [P1_urgent] My internet has been down for 6 hours, this is urgent!
  [P1_urgent] Router blinking red, no connectivity since morning, please fix ASAP
  [P2_medium] Billing amount seems slightly higher this month, can you check?
  [P2_medium] Would like to upgrade my plan sometime next week
  [P3_low] Just checking my current data balance please.
  [P3_low] Where can I download the invoice PDF from the app?


---
## Section 2: Concept Notes

### 2.1 — What is LoRA?

Full fine-tuning updates every weight matrix $W \in \mathbb{R}^{d \times k}$ in the model. LoRA instead **freezes $W$** and learns a *low-rank update* $\Delta W$:

$$\Delta W = B A, \quad B \in \mathbb{R}^{d \times r}, \quad A \in \mathbb{R}^{r \times k}, \quad r \ll \min(d, k)$$

The forward pass becomes $h = Wx + \frac{\alpha}{r}(BAx)$ — the frozen base output, plus a scaled correction from two tiny matrices. Only $A$ and $B$ are trainable.

**Key hyperparameters:**
| Param | Meaning | Typical range |
|---|---|---|
| `r` | rank of the update — controls adapter capacity | 4–64 |
| `lora_alpha` | scaling factor; effective scale is `alpha/r` | usually `2×r` |
| `target_modules` | which weight matrices get a LoRA adapter | attention projections (`q_lin`, `v_lin`, ...) |
| `lora_dropout` | dropout on the LoRA path only | 0.0–0.1 |
| `bias` | whether biases are trainable (`none`/`all`/`lora_only`) | `none` (default) |

**Why `q_lin`/`v_lin` and not all four projections?** The original LoRA paper found adapting query and value projections captures most of the benefit at a fraction of the parameter cost of adapting all four (`q_lin`, `k_lin`, `v_lin`, `out_lin`). You'll verify this trade-off numerically in Task 6.

### 2.2 — Why LoRA starts as a mathematical no-op

`lora_A` is initialized with small random values (Kaiming-uniform); **`lora_B` is initialized to exactly zero**. So at step 0: $\Delta W = BA = B \cdot A = 0 \cdot A = 0$ — the adapter contributes nothing, and the model behaves identically to the frozen base model on the very first forward pass. Training then gradually moves $B$ away from zero. You'll verify this directly by inspecting gradients in Task 4.

### 2.3 — What is QLoRA?

QLoRA = LoRA + **4-bit quantization of the frozen base model**. The base weights are stored in 4-bit NormalFloat (`nf4`) instead of 32-bit float — roughly an 8× memory reduction on the frozen weights — while the LoRA adapters themselves still train in higher precision (`bnb_4bit_compute_dtype`, typically `float16`). This is what makes fine-tuning multi-billion-parameter models possible on a single consumer GPU.

**Important limitation for today:** 4-bit loading requires an actual CUDA GPU. You can build and inspect a `BitsAndBytesConfig` object on CPU (Task 5), but loading a model *with* it requires Colab's T4 runtime — DistilBERT is small enough that QLoRA isn't really necessary for it, but the mechanics you set up today are identical to what you'd use for a 7B model in later work.

### 2.4 — Real-world use (why this matters for freelance/client work)

- **Storage:** one frozen base model + many small task-specific adapters (a few MB each) instead of a full checkpoint per task/client (hundreds of MB to GB).
- **Cost:** training only ~1% of parameters means smaller optimizer state (Adam keeps 2 extra values per trainable param) — this is often the difference between "fits in Colab free tier" and "needs a paid GPU."
- **Deployment:** adapters can be swapped in/out of the same base model at inference time — one deployed base model can serve multiple clients' fine-tuned behavior.
- **Interview framing:** *"We fine-tune a shared base model with LoRA adapters per client instead of maintaining N full model copies — this cuts our storage cost by ~90x and lets us onboard a new client's custom model in minutes instead of hours of retraining."*


---
## Section 3: Practice Guide

Complete Tasks 1–6 in order. Every task's code must actually run — read your printed output before writing any NRA cell. **Do not hardcode numbers from this notebook's markdown into your NRA cells — copy them from your own executed output.**

### Task 1 (15 pts) — Environment Setup & Baseline Parameter Count
Install/verify `peft`, `bitsandbytes`, `accelerate`. Load `distilbert-base-uncased` with `num_labels=3`. Print total vs. trainable parameters for the **unmodified** base model (should be 100% trainable — nothing is frozen yet).

### Task 2 (20 pts) — Apply LoRA and Verify the Parameter Drop
Build a `LoraConfig` (`r=8`, `lora_alpha=16`, `target_modules=["q_lin","v_lin"]`, `lora_dropout=0.05`, `bias="none"`, `task_type=TaskType.SEQ_CLS`). Wrap the model with `get_peft_model`. Call `.print_trainable_parameters()` and report the exact trainable count and percentage.

### Task 3 (20 pts) — Find the Hidden Fully-Trainable Layers
Inspect `named_parameters()` for everything with `requires_grad=True`. You will find LoRA `A`/`B` matrices **and** the `pre_classifier`/`classifier` head listed as `modules_to_save`. Explain why PEFT does this automatically for `TaskType.SEQ_CLS`.

### Task 4 (20 pts) — Verify Gradient Flow (the init no-op, proven)
Run one forward + backward pass on 2 sample tickets from Raw Data. Confirm: (a) a frozen, non-target layer's weight has `grad is None`; (b) `lora_A`'s gradient is zero at init; (c) `lora_B`'s gradient is non-zero at init. Connect this to Concept Note 2.2.

### Task 5 (15 pts) — Build a QLoRA (4-bit) Config
Construct a `BitsAndBytesConfig` (`load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_compute_dtype=torch.float16`, `bnb_4bit_use_double_quant=True`). Print it. Compute the estimated FP32 memory footprint of the full 67M-parameter model vs. its 4-bit footprint.

### Task 6 (10 pts) — Comparison Table + NRA
Build and print a comparison table: full fine-tune vs. LoRA r=8 (`q_lin`,`v_lin`) vs. LoRA r=8 (all 4 projections) vs. LoRA r=16 (`q_lin`,`v_lin`). Write one NRA recommending which config to carry into Day 209's actual training run.

### Bonus (10★) — Storage Economics
Using Task 2's trainable count, estimate the on-disk size of a saved LoRA adapter (fp32, 4 bytes/param) vs. a full model checkpoint. State the storage reduction factor.


---
## Your Working Area

Do your Task 1–6 work below. Keep one section per task. Answer Key and Scoring Rubric follow at the end — do not peek before attempting each task.


In [2]:
# ── SETUP: Install/upgrade PEFT, accelerate, bitsandbytes, and torchao ──
# Goal: Prepare the environment for LoRA/QLoRA experimentation.
# Method: Install the required libraries, upgrade torchao to avoid version conflict,
#         and check for GPU availability.

!pip install -q peft accelerate bitsandbytes
!pip install -q --upgrade torchao   # Fixes ImportError: incompatible torchao version

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU — Tasks 1-4 and 6 work fine; Task 5's config builds but won't load a model in 4-bit without a GPU runtime.")

CUDA available: True
GPU: Tesla T4


### Task 1 — Environment Setup & Baseline Parameter Count (15 pts)

In [3]:
# ── TASK 1 (15 pts): Baseline parameter count ──────────────────────────
# Goal: Load the base model, print its architecture facts, and report
#       the number of trainable parameters (should be 100%).
# Method: Load distilbert-base-uncased with num_labels=3, define a helper
#         to count trainable vs total params, and print the results.

from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 3

# Load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_params(m):
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    return trainable, total

trainable_baseline, total_baseline = count_params(model)
print(f"Baseline (no LoRA) — trainable: {trainable_baseline:,} | total: {total_baseline:,} | pct trainable: {100*trainable_baseline/total_baseline:.4f}%")
print(f"Vocab size: {tokenizer.vocab_size:,} | hidden layers: {model.config.n_layers} | hidden size: {model.config.dim} | attn heads: {model.config.n_heads}")



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline (no LoRA) — trainable: 66,955,779 | total: 66,955,779 | pct trainable: 100.0000%
Vocab size: 30,522 | hidden layers: 6 | hidden size: 768 | attn heads: 12


In [ ]:
# Number: trainable_baseline = 66,955,779 (100%).
# Reason: No PEFT config applied yet – every parameter in the model is trainable by default.
# Action: Full fine-tuning would update ~67M parameters; on limited hardware this would be slow.
#         The next step is to apply LoRA to drastically reduce the trainable count.

### Task 2 — Apply LoRA and Verify the Parameter Drop (20 pts)

In [4]:
# ── TASK 2 (20 pts): Apply LoRA and verify parameter drop ─────────────
# Goal: Build a LoRA config, wrap the model with PEFT, and print the
#       new trainable parameter count and percentage.
# Method: Use LoraConfig with r=8, alpha=16, target_modules=[q_lin, v_lin],
#         then call get_peft_model and print_trainable_parameters.

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_lin", "v_lin"],
    bias="none",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable_lora, total_lora = count_params(peft_model)
print(f"\nLoRA-wrapped — trainable: {trainable_lora:,} | total: {total_lora:,} | pct trainable: {100*trainable_lora/total_lora:.4f}%")
print(f"Reduction factor vs full fine-tune: {trainable_baseline/trainable_lora:.1f}x fewer trainable params")


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936

LoRA-wrapped — trainable: 740,355 | total: 67,696,134 | pct trainable: 1.0936%
Reduction factor vs full fine-tune: 90.4x fewer trainable params


In [ ]:
# Number: trainable_lora = 740,355 (1.0936% of total).
# Reason: LoRA freezes W and adds low-rank BA adapters only to q_lin and v_lin,
#         so only the adapter matrices and the classification head are trainable.
# Action: With only ~0.74M trainable params, we can fine-tune much faster and on cheaper hardware.
#         This enables adapter swapping for multiple clients without storing full checkpoints.

### Task 3 — Find the Hidden Fully-Trainable Layers (20 pts)

In [5]:
# ── TASK 3 (20 pts): Identify all trainable parameter groups ──────────
# Goal: Inspect named_parameters() to find every group with requires_grad=True.
#       Identify the non-LoRA layers that are also trainable (pre_classifier, classifier).
# Method: Loop over named_parameters, filter by requires_grad, and collect
#         unique layer patterns (generalise layer indices to N).

import re

seen_patterns = {}
for name, p in peft_model.named_parameters():
    if p.requires_grad:
        # Replace layer numbers with N so we see distinct groups
        generalized = re.sub(r"\.\d+\.", ".N.", name)
        if generalized not in seen_patterns:
            seen_patterns[generalized] = tuple(p.shape)

print("Distinct trainable parameter groups (layer index generalized):")
for pattern, shape in seen_patterns.items():
    print(f"  {pattern}  {shape}")



Distinct trainable parameter groups (layer index generalized):
  base_model.model.distilbert.transformer.layer.N.attention.q_lin.lora_A.default.weight  (8, 768)
  base_model.model.distilbert.transformer.layer.N.attention.q_lin.lora_B.default.weight  (768, 8)
  base_model.model.distilbert.transformer.layer.N.attention.v_lin.lora_A.default.weight  (8, 768)
  base_model.model.distilbert.transformer.layer.N.attention.v_lin.lora_B.default.weight  (768, 8)
  base_model.model.pre_classifier.modules_to_save.default.weight  (768, 768)
  base_model.model.pre_classifier.modules_to_save.default.bias  (768,)
  base_model.model.classifier.modules_to_save.default.weight  (3, 768)
  base_model.model.classifier.modules_to_save.default.bias  (3,)


In [ ]:
# Explanation (as comment)
# The two non-LoRA layers that are trainable are:
#   - pre_classifier (linear layer before classifier head)
#   - classifier (output layer for 3 classes)
# PEFT automatically adds these to `modules_to_save` for TaskType.SEQ_CLS because
# they are newly initialised for the classification task and were not part of the
# original pretrained checkpoint – they must be fully trained to adapt to our labels.

### Task 4 — Verify Gradient Flow: the Init No-Op, Proven (20 pts)

In [6]:
# ── TASK 4 (20 pts): Verify gradient flow at initialisation ──────────
# Goal: Run one forward+backward pass and inspect gradients to confirm:
#        (a) frozen non-target layer has grad=None
#        (b) lora_A grad is zero (since B is zero at init)
#        (c) lora_B grad is non-zero
# Method: Tokenise two sample tickets, compute loss, call backward(),
#         then inspect gradients of specific parameters.

# Use the first two sample tickets from Raw Data
encoded = tokenizer([t for t, _ in SAMPLE_TICKETS[:2]], padding=True, truncation=True, return_tensors="pt")
labels = torch.tensor([lbl for _, lbl in SAMPLE_TICKETS[:2]])

# Forward pass
output = peft_model(**encoded, labels=labels)
print("Logits shape:", tuple(output.logits.shape))
print("Loss:", round(output.loss.item(), 4))

# Backward pass
output.loss.backward()

# Inspect gradients for specific layers
frozen_grad = None
lora_A_grad = None
lora_B_grad = None
for name, p in peft_model.named_parameters():
    if "layer.0.attention.k_lin.weight" in name and "lora" not in name:
        frozen_grad = p.grad
    if "layer.0.attention.q_lin.lora_A" in name:
        lora_A_grad = p.grad
    if "layer.0.attention.q_lin.lora_B" in name:
        lora_B_grad = p.grad

print("\nFrozen k_lin (non-LoRA-target) grad is None:", frozen_grad is None)
print("lora_A grad exists:", lora_A_grad is not None, "| abs sum:", float(lora_A_grad.abs().sum()) if lora_A_grad is not None else None)
print("lora_B grad exists:", lora_B_grad is not None, "| abs sum:", float(lora_B_grad.abs().sum()) if lora_B_grad is not None else None)



[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Logits shape: (2, 3)
Loss: 1.0542

Frozen k_lin (non-LoRA-target) grad is None: True
lora_A grad exists: True | abs sum: 0.0
lora_B grad exists: True | abs sum: 0.25509607791900635


In [ ]:
# Number: lora_A abs‑sum = 0.0, lora_B abs‑sum = 0.255096.
# Reason: B is initialised to zero, so ΔW = B*A = 0; gradients flow through B only,
#         while A receives no gradient because it is multiplied by zero.
# Action: After one optimizer step, B will become non-zero and A will start receiving
#         gradients; this asymmetry is only at the very first step.

### Task 5 — Build a QLoRA (4-bit) Config (15 pts)

In [7]:
# ── TASK 5 (15 pts): Build QLoRA (4-bit) configuration ──────────────
# Goal: Create a BitsAndBytesConfig for 4-bit loading and estimate memory savings.
# Method: Instantiate BitsAndBytesConfig with load_in_4bit=True, nf4, float16,
#         double quant. Compute FP32 vs 4-bit footprints from total_baseline.

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
print(bnb_config)

# Memory estimates (frozen base model only)
fp32_mb = total_baseline * 4 / (1024 ** 2)
int4_mb = total_baseline * 0.5 / (1024 ** 2)  # 4 bits = 0.5 bytes per param
print(f"\nEstimated FP32 footprint (full {total_baseline:,} params): {fp32_mb:.1f} MB")
print(f"Estimated 4-bit footprint (same param count):            {int4_mb:.1f} MB")
print(f"Approx reduction: {fp32_mb/int4_mb:.1f}x")



BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


Estimated FP32 footprint (full 66,955,779 params): 255.4 MB
Estimated 4-bit footprint (same param count):            31.9 MB
Approx reduction: 8.0x


In [ ]:
# ── Task 5 NRA ──────────────────────────────────────────────────────────
# Number: FP32 = 255.4 MB, 4‑bit = 31.9 MB (8x reduction).
#
# Reason: Only the frozen base weights are quantised; the LoRA adapters
#         remain in fp32. This is safe because the frozen base weights are
#         never updated by gradient descent – their numerical precision
#         only affects forward‑pass fidelity, not gradient accuracy.
#         The LoRA adapters are being updated, so they need higher
#         precision to accumulate small gradient steps correctly.
#
# Action: With a T4's 16 GB VRAM, full fine‑tuning of models > ~4B
#         parameters becomes infeasible (4B × 4 bytes = 16 GB just for
#         weights). QLoRA's 4‑bit loading extends that ceiling to ~32B
#         parameters, assuming the same VRAM budget. At ~7B parameters,
#         this is the difference between "doesn't fit" and "fine‑tunes
#         comfortably" on a T4 – exactly the scenario this technique was
#         built for.

### Task 6 — Comparison Table + NRA (10 pts)

In [8]:
# ── TASK 6 (10 pts): Compare different LoRA configurations ────────────
# Goal: Build a table comparing trainable params for:
#        (1) Full fine-tune
#        (2) LoRA r=8, q_lin+v_lin
#        (3) LoRA r=8, all 4 projections
#        (4) LoRA r=16, q_lin+v_lin
# Method: Define a helper to create a model with a given config, compute counts,
#         and print a table with all rows.

def build_lora_model(r, alpha, targets):
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=alpha,
        lora_dropout=0.05,
        target_modules=targets,
        bias="none"
    )
    pm = get_peft_model(m, cfg)
    t, tot = count_params(pm)
    return t, tot

configs = [
    ("Full fine-tune (Month 9 style)", trainable_baseline, total_baseline),
    ("LoRA r=8, q_lin+v_lin",  *build_lora_model(8, 16, ["q_lin","v_lin"])),
    ("LoRA r=8, all 4 proj",   *build_lora_model(8, 16, ["q_lin","k_lin","v_lin","out_lin"])),
    ("LoRA r=16, q_lin+v_lin", *build_lora_model(16, 32, ["q_lin","v_lin"])),
]

print(f"{'Config':<32}{'Trainable':>14}{'Total':>14}{'% Trainable':>13}")
for name, t, tot in configs:
    print(f"{name:<32}{t:>14,}{tot:>14,}{100*t/tot:>12.4f}%")



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Config                               Trainable         Total  % Trainable
Full fine-tune (Month 9 style)      66,955,779    66,955,779    100.0000%
LoRA r=8, q_lin+v_lin                  740,355    67,696,134      1.0936%
LoRA r=8, all 4 proj                   887,811    67,843,590      1.3086%
LoRA r=16, q_lin+v_lin                 887,811    67,843,590      1.3086%


In [ ]:
# Number: For DistilBERT, LoRA r=8 q_lin+v_lin gives ~1.09% trainable (740k params).
# Reason: This config balances capacity (enough to adapt to the 3‑class ticket priority task)
#         with efficiency (only 2 out of 4 attention projections, lowest parameter count).
# Action: I will carry forward the "LoRA r=8, q_lin+v_lin" config to Day 209.
#         If validation F1 is disappointing, I would first try r=16 with the same projections
#         before adding all 4 projections, to keep the adapter size small.

### Bonus (10★) — Storage Economics

In [9]:
# ── BONUS (10★): Storage economics calculation ──────────────────────
# Goal: Estimate the on‑disk size of the LoRA adapter vs the full model checkpoint.
# Method: Multiply trainable_lora (from Task 2) by 4 bytes (fp32) to get adapter size.
#         Use total_baseline * 4 for the full checkpoint. Print both and the reduction factor.

lora_adapter_mb = trainable_lora * 4 / (1024 ** 2)
full_checkpoint_mb = total_baseline * 4 / (1024 ** 2)
print(f"LoRA adapter file (fp32): {lora_adapter_mb:.2f} MB")
print(f"Full model checkpoint (fp32): {full_checkpoint_mb:.2f} MB")
print(f"Storage reduction factor: {full_checkpoint_mb/lora_adapter_mb:.1f}x")



LoRA adapter file (fp32): 2.82 MB
Full model checkpoint (fp32): 255.42 MB
Storage reduction factor: 90.4x


In [ ]:
# ── Bonus interview framing ────────────────────────────────────────────
# "With LoRA, we store just a ~3 MB adapter per client instead of a
#  ~255 MB full model checkpoint – that's a 90x storage reduction,
#  which means we can serve hundreds of client‑specific fine‑tuned
#  models from the same base model without buying extra disk."

---
## Section 5: Scoring Rubric (100 pts + 10★ bonus)

| Task | Points | Full marks require |
|---|---:|---|
| Task 1 — Env setup & baseline count | 15 | Correct trainable/total (both = total params, 100%), correct architecture facts printed, NRA present |
| Task 2 — LoRA applied | 20 | Correct `LoraConfig` args, correct printed trainable %, NRA mechanism correctly ties $\Delta W=BA$ to the specific reduction factor (not generic "LoRA is efficient") |
| Task 3 — Hidden trainable layers | 20 | Correctly identifies `pre_classifier`+`classifier` as `modules_to_save`; explanation is *why* (newly-initialized, not pretrained) not just *what* |
| Task 4 — Gradient flow verified | 20 | Correct None/zero/non-zero pattern reported from actual printed output; NRA Reason correctly invokes $B_{init}=0$ mechanism, not a vague "gradients flow through trainable params" |
| Task 5 — QLoRA config | 15 | Correct `BitsAndBytesConfig` args; MB estimates computed from *printed* param count, not from memory/assumed round numbers |
| Task 6 — Comparison + NRA | 10 | Table has all 4 rows with correct numbers; NRA Action names a specific config decision for Day 209, not "we'll see" |
| **Total** | **100** | |
| Bonus★ | 10 | Correct MB math from Task 2's own printed trainable count + one genuinely client-facing sentence (no jargon dump) |

**Standing deductions (apply across all tasks, per established Month 13 pattern):**
- Any NRA Number not copied from *your own* printed output this session → **-3** per instance
- Reason bleeding into Action/outcome language ("...which means it's better/faster/cheaper") instead of stating the causal mechanism → **-2** per instance
- Any claim about which layers are trainable made without first printing `named_parameters()` → **-5** (must verify, not assume — same discipline as Day 202's `RetryPolicy` exception list)

---
**Key Takeaway:** LoRA doesn't make training "a little more efficient" — it changes *what class of problem is even attemptable* on your hardware, because freezing 99% of the parameters means the optimizer state (which is often 2-3x the model size for Adam) shrinks by the same ~90x factor. That's the number a client actually cares about, not "% trainable" in the abstract.

**Interview framing:** *"I froze the base DistilBERT weights and added rank-8 LoRA adapters to the attention query/value projections — that took trainable parameters from ~67M to ~740K, a 90x reduction, while keeping the same 3-class classification head fully trainable since it's task-specific and wasn't pretrained. On a bigger model this is the difference between fine-tuning fitting in a free Colab GPU or not fitting at all."*
